# Part 3 (Enhanced v2): Time-Series ML — Predict Youth Unemployment

**New in v2**
- ✅ Save-to-CSV toggle for the best model’s forecast
- ✅ Country **multi-select** to batch-generate & save forecasts for several countries
- ✅ Keeps single-country compare view (LR, Ridge, RF) with dropdown

**Inputs**: `youth_unemployment_cleaned_final.csv` (Part 2) or `youth_unemployment_long_clean.csv` (Part 1)

## 1) Setup & Load Data

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from ipywidgets import interact, Dropdown, SelectMultiple, Checkbox, Button, HBox, VBox, Output
    from IPython.display import display
    WIDGETS_OK = True
except Exception:
    WIDGETS_OK = False

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

path_candidates = ['youth_unemployment_cleaned_final.csv', 'youth_unemployment_long_clean.csv']
df = None
for p in path_candidates:
    try:
        df = pd.read_csv(p)
        print('Loaded:', p)
        break
    except Exception:
        pass
if df is None:
    raise FileNotFoundError('Place the cleaned CSV here or run Parts 1 & 2 first.')

rename_map = {'Country Name':'country','Country Code':'country_code','Year':'year','Youth_Unemployment':'youth_unemp'}
df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})
df.head()

Loaded: youth_unemployment_cleaned_final.csv


,country,country_code,year,youth_unemp
0,Afghanistan,AFG,1991,10.312
1,Afghanistan,AFG,1992,10.259
2,Afghanistan,AFG,1993,10.154
3,Afghanistan,AFG,1994,10.099
4,Afghanistan,AFG,1995,10.099


## 2) Utilities

In [5]:
def make_features(df_country):
    df_country = df_country.sort_values('year').copy()
    for lag in [1,2,3]:
        df_country[f'lag_{lag}'] = df_country['youth_unemp'].shift(lag)
    df_country['roll_mean_3'] = df_country['youth_unemp'].rolling(3).mean()
    df_country['roll_mean_5'] = df_country['youth_unemp'].rolling(5).mean()
    df_feat = df_country.dropna().reset_index(drop=True)
    return df_feat

def split_train_test(df_feat, test_years=5):
    last_year = int(df_feat['year'].max())
    split_year = last_year - test_years
    train = df_feat[df_feat['year'] <= split_year].copy()
    test  = df_feat[df_feat['year'] >  split_year].copy()
    feature_cols = ['year','lag_1','lag_2','lag_3','roll_mean_3','roll_mean_5']
    return train, test, feature_cols

def evaluate(y_true, y_pred):
    """Return MAE, RMSE, MAPE with compatibility for older scikit-learn.
    - RMSE: uses squared=False if available, else sqrt of MSE
    - MAPE: ignores division by zero by masking zeros in y_true
    """
    mae  = mean_absolute_error(y_true, y_pred)
    try:
        rmse = mean_squared_error(y_true, y_pred, squared=False)
    except TypeError:
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    y_true_s = pd.Series(y_true).replace(0, np.nan)
    mape = (np.abs((pd.Series(y_true) - pd.Series(y_pred)) / y_true_s)).mean(skipna=True) * 100
    return mae, rmse, mape

def forecast_next_years(history, model, feature_cols, years=5):
    history = history.sort_values('year').copy()
    future_rows = []
    for _ in range(years):
        target_year = int(history['year'].max()) + 1
        last = history.tail(5)
        lag_1 = last.iloc[-1]['youth_unemp'] if len(last)>=1 else np.nan
        lag_2 = last.iloc[-2]['youth_unemp'] if len(last)>=2 else np.nan
        lag_3 = last.iloc[-3]['youth_unemp'] if len(last)>=3 else np.nan
        roll_mean_3 = history['youth_unemp'].tail(3).mean() if len(history)>=3 else np.nan
        roll_mean_5 = history['youth_unemp'].tail(5).mean() if len(history)>=5 else np.nan
        row_feat = {'year':target_year,'lag_1':lag_1,'lag_2':lag_2,'lag_3':lag_3,'roll_mean_3':roll_mean_3,'roll_mean_5':roll_mean_5}
        for k,v in row_feat.items():
            if pd.isna(v):
                row_feat[k] = history['youth_unemp'].mean()
        X = pd.DataFrame([row_feat])[feature_cols]
        y_next = model.predict(X)[0]
        future_rows.append({'year':target_year,'youth_unemp_pred':y_next})
        history = pd.concat([history, pd.DataFrame({'year':[target_year],'youth_unemp':[y_next]})], ignore_index=True)
    return pd.DataFrame(future_rows)


## 3) Single-Country Comparison + Save Toggle

In [6]:
def compare_and_optionally_save(country, save_csv=False):
    df_c = df[df['country'] == country].copy()
    if df_c.empty:
        raise ValueError(f'{country} not in dataset')
    df_feat = make_features(df_c)
    if df_feat.shape[0] < 12:
        raise ValueError('Not enough rows after feature creation. Try another country.')
    train, test, feature_cols = split_train_test(df_feat, test_years=5)
    X_train, y_train = train[feature_cols], train['youth_unemp']
    X_test,  y_test  = test[feature_cols],  test['youth_unemp']

    models = {
        'LinearRegression': LinearRegression(),
        'Ridge(alpha=1.0)': Ridge(alpha=1.0),
        'RandomForest(n=300)': RandomForestRegressor(n_estimators=300, random_state=42)
    }
    results = []
    preds = {}
    for name, m in models.items():
        m.fit(X_train, y_train)
        p = m.predict(X_test)
        preds[name] = p
        mae, rmse, mape = evaluate(y_test, p)
        results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape})

    res_df = pd.DataFrame(results).sort_values('RMSE')
    best_name = res_df.iloc[0]['model']
    best_model = models[best_name]

    # Plot: test actual vs predictions
    plt.figure()
    plt.plot(test['year'], y_test.values, label='Actual', linewidth=2)
    for name, p in preds.items():
        plt.plot(test['year'], p, label=name)
    plt.title(f'{country}: Test — Actual vs Predicted (5-year holdout)')
    plt.xlabel('Year'); plt.ylabel('Youth Unemployment (%)'); plt.legend(); plt.tight_layout(); plt.show()

    # Forecast next 5 with best model
    future = forecast_next_years(df_c[['year','youth_unemp']].dropna(), best_model, feature_cols, years=5)

    plt.figure()
    plt.plot(df_c['year'], df_c['youth_unemp'], label='Historical')
    plt.plot(test['year'], preds[best_name], label=f'Test {best_name}')
    plt.plot(future['year'], future['youth_unemp_pred'], label='Forecast (Next 5)')
    plt.title(f'{country}: Historical, Best Test Prediction, Forecast')
    plt.xlabel('Year'); plt.ylabel('Youth Unemployment (%)'); plt.legend(); plt.tight_layout(); plt.show()

    if save_csv:
        out_path = f'forecast_{country.replace(" ","_")}.csv'
        future.assign(country=country).to_csv(out_path, index=False)
        print('Saved forecast →', out_path)

    return res_df, best_name, future

### Interactive Controls

In [7]:
if WIDGETS_OK:
    countries = sorted(df['country'].dropna().unique().tolist())
    dd = Dropdown(options=countries, value='Nigeria', description='Country:')
    save_ck = Checkbox(value=False, description='Save best forecast to CSV')
    out = Output()

    def on_change(change=None):
        with out:
            out.clear_output()
            res_df, best_name, future = compare_and_optionally_save(dd.value, save_csv=save_ck.value)
            display(res_df)
            display(future)

    dd.observe(on_change, names='value')
    save_ck.observe(on_change, names='value')
    display(HBox([dd, save_ck]))
    display(out)
    on_change()
else:
    print('Widgets not available here — running once for Nigeria with save_csv=False')
    res_df, best_name, future = compare_and_optionally_save('Nigeria', save_csv=False)
    display(res_df); display(future)

Output()

## 4) Batch Forecasts for Multiple Countries

In [8]:
def batch_forecast(countries, years=5):
    all_rows = []
    for country in countries:
        try:
            df_c = df[df['country'] == country].copy()
            df_feat = make_features(df_c)
            if df_feat.shape[0] < 12:
                print(f'Skipping {country}: insufficient history after features')
                continue
            train, test, feature_cols = split_train_test(df_feat, test_years=5)
            X_train, y_train = train[feature_cols], train['youth_unemp']
            # Use a robust default model for batch
            model = RandomForestRegressor(n_estimators=300, random_state=42)
            model.fit(X_train, y_train)
            future = forecast_next_years(df_c[['year','youth_unemp']].dropna(), model, feature_cols, years=years)
            future = future.assign(country=country)
            all_rows.append(future)
        except Exception as e:
            print(f'Error for {country}:', e)
            continue
    if all_rows:
        out_df = pd.concat(all_rows, ignore_index=True)
        out_df.to_csv('batch_forecasts.csv', index=False)
        print('Saved batch forecasts → batch_forecasts.csv')
        return out_df
    else:
        print('No forecasts produced.')
        return pd.DataFrame()


### Batch UI

In [9]:
if WIDGETS_OK:
    countries = sorted(df['country'].dropna().unique().tolist())
    sel = SelectMultiple(options=countries, value=tuple(['Nigeria','South Africa','Kenya']), description='Countries')
    run_btn = Button(description='Run Batch Forecasts', button_style='success')
    out2 = Output()

    def on_click(btn):
        with out2:
            out2.clear_output()
            out_df = batch_forecast(list(sel.value), years=5)
            display(out_df.head(20))

    run_btn.on_click(on_click)
    display(VBox([sel, run_btn, out2]))
else:
    print('Widgets not available — running batch for Nigeria, South Africa, Kenya...')
    out_df = batch_forecast(['Nigeria','South Africa','Kenya'], years=5)
    display(out_df.head(20))